# 00.1 · Why machine learning? Rules vs learning from data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/00-big-picture/01-why-machine-learning/lesson.ipynb)

Read the [README](README.md) first. This notebook is the "In code" and "On real data" parts of the lesson.

**What you will do**

1. Load the eight ShopWave orders from the README.
2. Write your hand rule as a function and score it.
3. Let a tiny decision tree learn its own rules from the same examples.
4. Compare, then scale up to 400 orders where hand rules stop working.
5. Glimpse the real data.

You do **not** need to understand the scikit-learn code yet. Watch what it does.

## 1. The eight orders

In [1]:
import pandas as pd

orders = pd.DataFrame({
    "total":          [45, 620, 310, 880, 95, 540, 430, 70],
    "account_days":   [400, 3, 120, 1, 2, 800, 5, 30],
    "items":          [2, 9, 4, 12, 1, 6, 7, 3],
    "country_match":  [1, 0, 1, 1, 1, 0, 0, 1],   # 1 = shipping country matches card country
    "fraud":          [0, 1, 0, 1, 0, 0, 1, 0],
})
orders

,total,account_days,items,country_match,fraud
0,45,400,2,1,0
1,620,3,9,0,1
2,310,120,4,1,0
3,880,1,12,1,1
4,95,2,1,1,0
5,540,800,6,0,0
6,430,5,7,0,1
7,70,30,3,1,0


## 2. Your hand rule

From the README, the best rule by eye was *"total over 400 **and** account under 7 days"*. Edit the function if you found a different one.

In [2]:
def hand_rule(row):
    return int(row["total"] > 400 and row["account_days"] < 7)

orders["hand_pred"] = orders.apply(hand_rule, axis=1)
orders[["total", "account_days", "fraud", "hand_pred"]]

,total,account_days,fraud,hand_pred
0,45,400,0,0
1,620,3,1,1
2,310,120,0,0
3,880,1,1,1
4,95,2,0,0
5,540,800,0,0
6,430,5,1,1
7,70,30,0,0


In [3]:
accuracy = (orders["hand_pred"] == orders["fraud"]).mean()
print(f"Hand rule accuracy on 8 orders: {accuracy:.0%}")

Hand rule accuracy on 8 orders: 100%


## 3. Let the computer find the rule

`DecisionTreeClassifier` is an algorithm that searches for the best if/else questions, exactly what you did by eye. `max_depth=2` means "at most two questions deep", so the result stays readable.

- `X` = the feature columns (what the model may look at)
- `y` = the target column (what it must predict)
- `.fit(X, y)` = learn

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["total", "account_days", "items", "country_match"]
X = orders[features]
y = orders["fraud"]

tree = DecisionTreeClassifier(max_depth=2, random_state=0)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- items <= 6.50
|   |--- class: 0
|--- items >  6.50
|   |--- class: 1



Read the printed tree top to bottom. Each `|---` line is a question; `class: 1` means "fraud".

Surprise: the tree did not pick your rule. It found that a single question, **items > 6.5**, separates all eight orders perfectly. It tried every column and every cut-point and kept the one that split the classes best, and this one needed only one question.

Is the tree right and you wrong? Neither. With eight examples, *many* different rules fit perfectly, and there is no way to tell which one describes real fraud. This is the first lesson about data: **a model can only be as good as the number and variety of examples it learns from.** Let us give it more.

In [5]:
orders["tree_pred"] = tree.predict(X)
print(f"Tree accuracy on 8 orders: {(orders['tree_pred'] == orders['fraud']).mean():.0%}")
orders[["total", "account_days", "fraud", "hand_pred", "tree_pred"]]

Tree accuracy on 8 orders: 100%


,total,account_days,fraud,hand_pred,tree_pred
0,45,400,0,0,0
1,620,3,1,1,1
2,310,120,0,0,0
3,880,1,1,1,1
4,95,2,0,0,0
5,540,800,0,0,0
6,430,5,1,1,1
7,70,30,0,0,0


## 4. Scale up: 400 orders

Eight rows is a toy. Below we generate 400 orders that follow the *same hidden pattern* with added noise, the way real fraud does. Now compare the hand rule with a tree that learns from the data.

(The generator is not something you need to read. In the real world this data comes from the orders table.)

In [6]:
import numpy as np
rng = np.random.default_rng(0)

def make_orders(n):
    """Generate n ShopWave orders with the same hidden pattern as the 8 above."""
    total = rng.gamma(2.0, 150, n).round(0)
    account_days = rng.integers(1, 1000, n)
    account_days = np.where(rng.random(n) < 0.25, rng.integers(1, 15, n), account_days)  # a quarter are new accounts
    items = np.clip((total / 70 + rng.normal(0, 1.5, n)).round(), 1, None).astype(int)
    country_match = rng.choice([0, 1], n, p=[0.15, 0.85])
    # hidden truth (the model never sees this line): big order from a new account,
    # OR a very big order shipping to a country that does not match the card.
    # 3% of labels are flipped at random, because real data is never clean.
    truth = ((total > 300) & (account_days < 10)) | ((country_match == 0) & (total > 500))
    fraud = (truth ^ (rng.random(n) < 0.03)).astype(int)
    return pd.DataFrame({"total": total, "account_days": account_days,
                         "items": items, "country_match": country_match, "fraud": fraud})

many = make_orders(400)
print(many.shape)
print("fraud rate:", many["fraud"].mean().round(3))
many.head()

(400, 5)
fraud rate: 0.11


,total,account_days,items,country_match,fraud
0,275.0,281,2,1,0
1,396.0,227,4,1,0
2,160.0,208,2,1,0
3,597.0,33,10,1,0
4,137.0,3,3,1,0


In [7]:
many["hand_pred"] = many.apply(hand_rule, axis=1)
hand_acc = (many["hand_pred"] == many["fraud"]).mean()

X_many, y_many = many[features], many["fraud"]
tree_many = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_many, y_many)
tree_acc = (tree_many.predict(X_many) == y_many).mean()

print(f"Hand rule accuracy: {hand_acc:.1%}")
print(f"Learned tree accuracy: {tree_acc:.1%}")
print()
print(export_text(tree_many, feature_names=features))

Hand rule accuracy: 90.8%
Learned tree accuracy: 96.5%

|--- account_days <= 9.50
|   |--- total <= 297.00
|   |   |--- account_days <= 8.50
|   |   |   |--- class: 0
|   |   |--- account_days >  8.50
|   |   |   |--- class: 0
|   |--- total >  297.00
|   |   |--- class: 1
|--- account_days >  9.50
|   |--- country_match <= 0.50
|   |   |--- total <= 511.00
|   |   |   |--- class: 0
|   |   |--- total >  511.00
|   |   |   |--- class: 1
|   |--- country_match >  0.50
|   |   |--- account_days <= 949.00
|   |   |   |--- class: 0
|   |   |--- account_days >  949.00
|   |   |   |--- class: 0



Three things to notice:

1. **Check the baseline first.** About 11% of orders are fraud, so a "rule" that flags nothing scores 89%. The hand rule is barely better than that. Always ask what the dumbest possible answer scores before you admire a number.
2. **The tree does clearly better**, because it searched thresholds and combinations you would never have tried, and it found the country-mismatch pattern that your rule ignored.
3. **The tree is still readable.** Machine learning does not have to be a black box; for many business problems you can print the learned rules and check them with the operations team.

A warning you will understand fully in Module 04: we scored the tree on the *same* orders it learned from. That flatters it. Real evaluation uses orders the model has never seen.

## 5. A glimpse of the real data

ShopWave's "real" data in this course is the Online Retail II dataset: about a million actual transactions from a UK online retailer over two years. The cell below downloads it (about 45 MB, once) and shows the first rows.

You will spend Module 01 learning to handle data like this. For now, just look at the size and ask yourself how far hand-written rules would get you.

In [8]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail

path = ensure_online_retail()
retail = pd.read_excel(path, sheet_name="Year 2010-2011")
print(f"{len(retail):,} transactions, {retail['Customer ID'].nunique():,} customers, "
      f"{retail['StockCode'].nunique():,} products, {retail['Country'].nunique()} countries")
retail.head()

541,910 transactions, 4,372 customers, 4,070 products, 38 countries


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Quiz answers

1. **Learning** means the program's performance at a task improves as it sees more examples (experience), without being given the rules explicitly.
2. House **size** is the feature (input); **price** is the target (what we predict).
3. **False.** When the rule is known, must be guaranteed, or no examples exist, rules are better. ML wins when nobody can write the rule but examples are plentiful.
4. Each new fraud pattern needed a new rule; rules began to overlap and contradict; nobody could predict the function's output. The approach does not scale when the rules are unknown and changing.
5. Any of: VAT calculation, "never ship to sanctioned countries", age verification, refund policy. Known, fixed, must-be-guaranteed rules.

**Next:** [00.2 The AI family tree](../02-ai-family-tree/)